<a href="https://colab.research.google.com/github/kalyandrug/rk/blob/main/clinical_data_diabetic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

RANDOM_STATE = 42

In [ ]:
#upload file
from google.colab import files

# Upload the file auto csv file
uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"Uploaded file: {filename}")

In [ ]:
# -----------------------------------------------------
# 1. Load data
# -----------------------------------------------------
# Option A: use your own auto MPG CSV (uncomment and edit path)
df = pd.read_csv(filename, na_values='?', comment='\t', sep=',', skipinitialspace=True)

In [ ]:
#display datasets
print(df)

In [ ]:
df.describe()

In [ ]:
# 1. Identify missing values in the dataset.
print("Missing values in the dataset:\n", df.isnull().sum())

In [ ]:
# 3. REMOVE EXACT DUPLICATE ROWS
# ---------------------------------------------------------------------------
dupes_before = df.duplicated().sum()
df = df.drop_duplicates()
print(f"\nDropped {dupes_before} exact duplicate rows. New shape: {df.shape}")

In [ ]:
# 4. CONVERT DISGUISED ZEROS TO NaN
#    These columns cannot legitimately be 0 in a living patient.
#    'pregnancies' and 'outcome' are excluded — 0 pregnancies and 0 outcome
#    (non-diabetic) are valid real values.
# ---------------------------------------------------------------------------
zero_as_missing_cols = ["glucose", "blood_pressure", "skin_thickness", "insulin", "bmi"]

print("\nZero-value counts per column BEFORE conversion (likely disguised missing data):")
print((df[zero_as_missing_cols] == 0).sum())

for col in zero_as_missing_cols:
    df[col] = df[col].replace(0, np.nan)

print("\nMissing value counts AFTER converting disguised zeros:")
print(df[zero_as_missing_cols].isna().sum())
print(f"\n'insulin' missing rate: {df['insulin'].isna().mean():.1%}")
print(f"'skin_thickness' missing rate: {df['skin_thickness'].isna().mean():.1%}")

In [ ]:
# ---------------------------------------------------------------------------
# 5. OUTLIER / RANGE CHECK
#    Flag biologically implausible values that survived (too high, not just
#    too low). This dataset is fairly clean, but always verify.
# ---------------------------------------------------------------------------
range_checks = {
    "glucose": (40, 300),          # mg/dL
    "blood_pressure": (30, 180),   # mmHg (diastolic)
    "bmi": (10, 70),
    "age": (18, 100),
}
print("\nOut-of-range value counts (values outside plausible clinical range):")
for col, (low, high) in range_checks.items():
    out_of_range = ((df[col] < low) | (df[col] > high)).sum()
    print(f"  {col}: {out_of_range} rows outside [{low}, {high}]")

In [ ]:

# ---------------------------------------------------------------------------
# 6. SAVE CLEANED (BUT NOT YET IMPUTED) DATASET
#    We deliberately do NOT impute here. Imputation should be fit on the
#    training split only, inside your modeling pipeline — not on the full
#    dataset — to avoid data leakage. This script hands off a clean,
#    correctly-typed, duplicate-free, properly-NaN-flagged CSV.
# ---------------------------------------------------------------------------
df.to_csv("diabetes_clinical_cleaned.csv", index=False)
print("\nSaved cleaned dataset to diabetes_clinical_cleaned.csv")
print("Final shape:", df.shape)
print("\nRemaining missing values (to be imputed inside train/test split during modeling):")
print(df.isna().sum())

In [ ]:
"""
Diabetes Prediction — End-to-End ML Pipeline
===============================================
Dataset: diabetes_clinical_cleaned.csv (768 patients, Pima Indians Diabetes)
Target : outcome (1 = diabetic, 0 = not diabetic)

This script assumes diabetes_data_cleanup.py has already been run — the
input CSV has disguised zeros already converted to NaN, and duplicates
already removed. This script handles imputation INSIDE the pipeline
(fit on train split only) to avoid data leakage, then trains and compares
Logistic Regression vs Random Forest.
"""

# ---------------------------------------------------------------------------
# 2. DEFINE FEATURES / TARGET
# ---------------------------------------------------------------------------
TARGET = "outcome"
feature_cols = [c for c in df.columns if c != TARGET]

X = df[feature_cols]
y = df[TARGET]

In [ ]:

# ---------------------------------------------------------------------------
# 3. TRAIN/TEST SPLIT (stratified — preserves ~35% diabetic ratio in both sets)
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]}  Test size: {X_test.shape[0]}")
